# RAG [Retrieval-Augmented Generation]

- Simple rag pipeline using langchain.
- Embedding Model used: `all-MiniLM-L6-v2` | `langchain_huggingface.HuggingFaceEmbeddings`
- LLM used: `gemini-2.5-flash` | `langchain_google_genai.ChatGoogleGenerativeAI`

---
Currently I am gonna implement by attaching my Resume pdf file to build the simple rag pipeline.

In [14]:
import os
import json
from dotenv import load_dotenv
# 
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser

from langchain_google_genai import ChatGoogleGenerativeAI


# 
load_dotenv()

True

Contants:

In [9]:
SAMPLES_DIR = os.path.abspath('samples')
RESUME_FILE_PTH = os.path.join(SAMPLES_DIR, 'my-resume.pdf')
# 
EMBEDDING_MODEL = 'all-MiniLM-L6-v2'
LLM_MODEL = 'gemini-2.5-flash'

### Load the Docuements & Store to the VectorDB:

Loader:

In [10]:
from langchain.document_loaders import PyPDFLoader


pdf_loader = PyPDFLoader(
    file_path=RESUME_FILE_PTH,
)

In [12]:
documents = pdf_loader.load()

print(json.dumps(documents[0].metadata, indent=True))

{
 "producer": "Canva",
 "creator": "Canva",
 "creationdate": "2025-04-24T11:25:03+00:00",
 "title": "SculpSoft - Resume",
 "moddate": "2025-04-24T11:25:02+00:00",
 "keywords": "DAGliug431E,BAERE5KljWo,0",
 "author": "KIRTAN GHELANI",
 "source": "e:\\00_SCULPTSOFT\\training-internship\\Training-Tasks---SculptSoft\\Gen AI - LLMs\\basics\\samples\\my-resume.pdf",
 "total_pages": 2,
 "page": 0,
 "page_label": "1"
}


Splitters

In [18]:
from langchain.text_splitter import RecursiveCharacterTextSplitter 

parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1500,
    chunk_overlap = 200
)

child_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 400,
    chunk_overlap = 80
)

Embeddings:

In [19]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model = EMBEDDING_MODEL
)

Vector Store Setup and data Load:

In [20]:
from langchain_chroma.vectorstores import Chroma
from langchain.storage import InMemoryStore

vector_store = Chroma(
    collection_name = 'resume_rag',
    embedding_function= embeddings
)

store = InMemoryStore()

### Retriever Building:

In [22]:
from langchain.retrievers import ParentDocumentRetriever

retriever = ParentDocumentRetriever(
    vectorstore=vector_store,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)


# Adding Documents to Retriever:
retriever.add_documents(documents=documents,
                        ids=None)

### LLM Initializing:

In [23]:
llm = ChatGoogleGenerativeAI(
    model = LLM_MODEL,
    temperature = 0.7,
    max_tokens = None,
    max_retries = 2
)

# RAG + LLM Pipeline Building:

In [24]:
TEMPLATE = """
Answer the question based only on the following context:
{context}

Question: {question}
"""

In [25]:
prompt = ChatPromptTemplate.from_template(TEMPLATE)

Chain Building:

In [27]:
from langchain_core.runnables import RunnablePassthrough


rag_pipeline = {
    "context" : retriever | (lambda docs: "\n\n".join(doc.page_content for doc in docs)),
    "question" : RunnablePassthrough() 
    } | prompt | llm | StrOutputParser()

### Testing [Retriever]:

In [40]:
child_docs = vector_store.similarity_search("grades")
print(child_docs[0].page_content)

Achievements
O t h 
SRM Institute of Science and Technology
B.Tech in Computer Science and Engineering.
CGPA: 9.34 (93.4%).
Relevant coursework: Software Engineering, AI, DSA, OOPS, DBMS, Applied programming, Analysis of algorithms. 
September 2020 - October 2024
+ 9 1 - 8 0 0 0 8 2 3 6 5 4 
 G i t h u b : g i t h u b . c o m / g h e l a n i k i r t a n


### Testing [RAG-Pipeline]:

In [42]:
Query = "How much did candidate scored in his university?"


res = rag_pipeline.invoke(Query)
print(res)

The candidate scored a CGPA of 9.34 (93.4%) in his university.


## CHAT-BOT:

In [47]:
print("Start Interacting with the RAG-Pipline:")
while True:
    query = input("Ask Question or 'quit' to exit...\n")
    
    if query.lower() == 'quit':
        break
    print(f"Question: {query}")
    res = rag_pipeline.invoke(query)
    print(f"""Answer: 
{res}""")
    print('-'*80)

Start Interacting with the RAG-Pipline:
Question: give me candidate name
Answer: 
KIRTAN YOGESHKUMAR GHELANI
--------------------------------------------------------------------------------
Question: What's his educational qualification?
Answer: 
Based on the context provided, his educational qualification is:

*   **B.Tech in Computer Science and Engineering** from SRM Institute of Science and Technology.
--------------------------------------------------------------------------------
Question: his grades?
Answer: 
His CGPA is 9.34 (93.4%).
--------------------------------------------------------------------------------
Question: List of technical skills of these candidates
Answer: 
Based on the provided context, here is a list of technical skills:

**Programming Languages:**
*   Python
*   C++
*   JavaScript
*   TypeScript
*   SQL
*   HTML
*   CSS & Tailwind CSS

**Domain-wise Skills:**
*   AI-ML
*   LLMs and RAGs
*   Software Management
*   Data Structures and Algorithms
*   Object 

---
By Kirtan Ghelani `@SculptSoft`